# BS Binary-Call Market-Making Backtest

**Market:** *Bitcoin Up or Down — April 20, 8:40PM-8:45PM ET*  (slug: `btc-updown-5m-1776732000`)

- Strike fixes at **2026-04-21 00:40 UTC** (= BTC spot at that moment)
- Resolves at **00:45 UTC**
- *Down* won → `Up` settles at **0.0**

## Strategy

On every clock tick during the 5-min window we compute the Black-Scholes binary-call fair value of the *Up* outcome:

$$
P(S_T > K) = \Phi(d_2),\quad d_2 = \frac{\ln(S/K) - \sigma^2 (T - t)/2}{\sigma\sqrt{T - t}}
$$

Then we post a two-sided resting limit quote at `fair ± half_spread`, cancelling and re-posting when fair moves more than `requote_threshold`. The ask is only quoted when we hold inventory (engine rejects naked shorts).

**BTC data:** Coinbase BTC-USD 1-min OHLC (linearly interpolated between minute opens). Higher-frequency public BTC sources are geo-restricted from the US; for a strategy this short-horizon, sub-minute BTC moves would matter — note the limitation.

**Run config:** σ_annual=0.6, half_spread=0.02, quote_size=25, tick=1s, initial_cash=$1000.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

## 1. Run the backtest

All the heavy lifting (Polymarket trade fetch, Coinbase BTC fetch with disk cache, strategy + engine) lives in `bs_mm_backtest.py`. The `main()` function returns a `BacktestResult` with the full metrics surface from the rewritten `backtester`.

In [ ]:
from examples.bs_mm_backtest import main

result = main(
    sigma_annual=0.6,
    half_spread=0.02,
    quote_size=25,
    max_inventory=500,
    requote_threshold=0.005,
    tick_interval="1s",
    initial_cash=1000.0,
    taker_fee_bps=0.0,
    maker_rebate_bps=0.0,
    show_plot=False,
)

## 2. Summary metrics

In [ ]:
import pandas as pd
pd.Series(result.summary())

## 3. Round-trip P&L distribution

In [ ]:
rt = pd.DataFrame(result.round_trips)
if not rt.empty:
    rt["entry"] = pd.to_datetime(rt["entry_ts"], unit="ms", utc=True)
    rt["exit"]  = pd.to_datetime(rt["exit_ts"],  unit="ms", utc=True)
    rt["hold_seconds"] = (rt["exit"] - rt["entry"]).dt.total_seconds()
    display(rt[["entry", "exit", "qty", "entry_price", "exit_price", "pnl", "hold_seconds"]].head(20))
rt["pnl"].describe() if not rt.empty else None

## 4. Plot — price + PnL + drawdown

In [ ]:
result.plot()

## 5. Quick parameter sweep

Vary half-spread to see how it interacts with adverse selection on this market.

In [ ]:
rows = []
for hs in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08]:
    r = main(
        sigma_annual=0.6, half_spread=hs, quote_size=25, max_inventory=500,
        requote_threshold=0.005, tick_interval="1s", initial_cash=1000.0,
        show_plot=False,
    )
    s = r.summary()
    rows.append({
        "half_spread": hs,
        "total_pnl": s["total_pnl"],
        "num_fills": s["num_fills"],
        "win_rate": s["win_rate"],
        "max_dd": s["max_drawdown"],
    })
sweep = pd.DataFrame(rows)
sweep